# Research Agent RL — Week 1: SFT on Kaggle (T4 GPU)

**Setup:** Before running, make sure you have:
- Accelerator set to **GPU T4 x1** (Settings → Accelerator)
- Internet access **ON** (Settings → Internet)

This notebook:
1. Clones the repo and installs dependencies
2. Builds the HotpotQA SFT dataset
3. Fine-tunes Qwen2.5-0.5B-Instruct with 4-bit QLoRA
4. Saves the adapter checkpoint

## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — enable T4 in Settings!')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install all required packages (takes ~2 min on Kaggle)
!pip install -q -r requirements.txt

In [ ]:
# Verify key packages loaded correctly
import torch
import transformers, trl, peft, bitsandbytes, datasets
print(f'torch       {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
print(f'transformers {transformers.__version__}')
print(f'trl          {trl.__version__}')
print(f'peft         {peft.__version__}')
print(f'bitsandbytes {bitsandbytes.__version__}')
print(f'datasets     {datasets.__version__}')

## 3. (Optional) Adjust config

Edit the cell below to override any `config.yaml` values without touching the file.

In [ ]:
import yaml

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# --- Tweak anything here ---
cfg['dataset']['train_size'] = 8000   # reduce to 2000 for a quick smoke-test
cfg['dataset']['val_size']   = 500
cfg['training']['num_train_epochs'] = 3
cfg['training']['report_to'] = 'none'
# ---------------------------

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config saved. Key settings:')
print(f"  model           : {cfg['model']['name']}")
print(f"  train_size      : {cfg['dataset']['train_size']}")
print(f"  val_size        : {cfg['dataset']['val_size']}")
print(f"  epochs          : {cfg['training']['num_train_epochs']}")
print(f"  lr              : {cfg['training']['learning_rate']}")
print(f"  LoRA r / alpha  : {cfg['lora']['r']} / {cfg['lora']['lora_alpha']}")

## 4. Build SFT dataset from HotpotQA

In [ ]:
# Downloads HotpotQA (~500 MB) and builds reasoning traces
# Expected runtime: ~4–6 min for 8k train + 500 val traces
!python data/prepare_sft_dataset.py

In [ ]:
# Spot-check the first trace
import json

with open('data/sft_traces/train.jsonl') as f:
    sample = json.loads(f.readline())

print('Question:', sample['question'])
print('Answer  :', sample['answer'])
print(f'Steps   : {len(sample["trace"])}')
print()
for i, step in enumerate(sample['trace'], 1):
    print(f'  Step {i}:', json.dumps(step))

## 5. Verify tokenization & instruction masking

In [ ]:
from transformers import AutoTokenizer
from data.sft_dataset import SFTTraceDataset

tokenizer = AutoTokenizer.from_pretrained(cfg['model']['name'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = SFTTraceDataset('data/sft_traces/train.jsonl', tokenizer, max_length=cfg['model']['max_seq_length'])
item = ds[0]

n_total  = len(item['input_ids'])
n_masked = (item['labels'] == -100).sum().item()
n_train  = n_total - n_masked

print(f'Sequence length : {n_total} tokens')
print(f'Masked (ignored): {n_masked} tokens  ({100*n_masked/n_total:.1f}%)')
print(f'Trained on      : {n_train} tokens  ({100*n_train/n_total:.1f}%)')
print()
# Decode the trained portion to confirm it is the assistant content
trained_ids = [id_.item() for id_, lbl in zip(item['input_ids'], item['labels']) if lbl != -100]
print('Trained text (first 300 chars):')
print(tokenizer.decode(trained_ids[:150]))

## 6. Load model (Qwen2.5-0.5B + 4-bit QLoRA)

In [ ]:
from sft.model import load_model_and_tokenizer

# Downloads model weights (~1.5 GB) on first run
model, tokenizer = load_model_and_tokenizer('config.yaml')

# Confirm VRAM usage
import torch
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
print(f'\nVRAM allocated : {allocated:.2f} GB')
print(f'VRAM reserved  : {reserved:.2f} GB')

## 7. Train

In [ ]:
import torch
from functools import partial
from pathlib import Path

from transformers import Trainer, TrainingArguments
from data.sft_dataset import SFTTraceDataset, collate_fn

train_dataset = SFTTraceDataset('data/sft_traces/train.jsonl', tokenizer,
                                 max_length=cfg['model']['max_seq_length'])
val_dataset   = SFTTraceDataset('data/sft_traces/val.jsonl',   tokenizer,
                                 max_length=cfg['model']['max_seq_length'])

tcfg = cfg['training']
output_dir = tcfg['output_dir']
Path(output_dir).mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=tcfg['num_train_epochs'],
    per_device_train_batch_size=tcfg['per_device_train_batch_size'],
    per_device_eval_batch_size=tcfg['per_device_eval_batch_size'],
    gradient_accumulation_steps=tcfg['gradient_accumulation_steps'],
    learning_rate=tcfg['learning_rate'],
    warmup_ratio=tcfg['warmup_ratio'],
    lr_scheduler_type=tcfg['lr_scheduler_type'],
    logging_steps=tcfg['logging_steps'],
    eval_steps=tcfg['eval_steps'],
    save_steps=tcfg['save_steps'],
    save_total_limit=tcfg['save_total_limit'],
    fp16=tcfg['fp16'],
    dataloader_num_workers=tcfg['dataloader_num_workers'],
    report_to=tcfg['report_to'],
    load_best_model_at_end=tcfg['load_best_model_at_end'],
    metric_for_best_model=tcfg['metric_for_best_model'],
    greater_is_better=tcfg['greater_is_better'],
    eval_strategy='steps',
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

pad_collate = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=pad_collate,
)

print(f'Training on {len(train_dataset)} samples, validating on {len(val_dataset)} samples')
trainer.train()

## 8. Save final checkpoint

In [ ]:
import os

final_dir = os.path.join(tcfg['output_dir'], 'final')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'Adapter saved → {final_dir}')
!ls -lh {final_dir}

## 9. Quick inference test

In [ ]:
import torch, json
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from data.sft_dataset import SYSTEM_PROMPT

# Reload from saved adapter for a clean test
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    cfg['model']['name'], quantization_config=bnb_config,
    device_map={'': 0}, trust_remote_code=True,
)
inf_model = PeftModel.from_pretrained(base, final_dir)
inf_model.eval()
inf_tok = AutoTokenizer.from_pretrained(final_dir, trust_remote_code=True)

test_question = 'What is the birthplace of the director of the 2010 film Inception?'

messages = [
    {'role': 'system',    'content': SYSTEM_PROMPT},
    {'role': 'user',      'content': f'Question: {test_question}'},
]
prompt = inf_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = inf_tok(prompt, return_tensors='pt').to(inf_model.device)

with torch.no_grad():
    output_ids = inf_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
        eos_token_id=inf_tok.eos_token_id,
    )

generated = inf_tok.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Model output:')
print(generated)

# Try to parse each line as JSON
print('\nParsed steps:')
for line in generated.strip().splitlines():
    line = line.strip()
    if line.startswith('Step'):
        line = line.split(':', 1)[-1].strip()
    try:
        step = json.loads(line)
        print(f"  action={step.get('action')!r:8s}  confidence={step.get('confidence')}")
    except json.JSONDecodeError:
        pass

## 10. (Optional) Push checkpoint to Hugging Face Hub

In [ ]:
# Fill in your HF token and repo name, then uncomment to push

# HF_TOKEN   = 'hf_...'
# HF_REPO_ID = 'your-username/qwen-research-agent-sft'

# from huggingface_hub import login
# login(token=HF_TOKEN)
# inf_model.push_to_hub(HF_REPO_ID)
# inf_tok.push_to_hub(HF_REPO_ID)
# print(f'Pushed to https://huggingface.co/{HF_REPO_ID}')